# Aggregate 6-config evaluation results

In [6]:
import json
import sys
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from experiments.evaluation.tool_accuracy import check_tool_accuracy, check_tool_precision

RUN_DIR = ROOT / 'data' / 'evaluation' / 'run_results'

## Step 1 — Run + judge jsonl pairs 

In [7]:
PAIRS = {
    'C1': (RUN_DIR / 'c1_20260429_163155.jsonl', RUN_DIR / 'c1_20260429_163155.judge.jsonl'),
    'C2': (RUN_DIR / 'c2_20260429_144635.jsonl', RUN_DIR / 'c2_20260429_144635.judge.jsonl'),
    'C3': (RUN_DIR / 'c3_20260429_081503.jsonl', RUN_DIR / 'c3_20260429_081503.judge.jsonl'),
    'C4': (RUN_DIR / 'c4_20260429_111924.jsonl', RUN_DIR / 'c4_20260429_111924.judge.jsonl'),
    'C5': (RUN_DIR / 'c5_20260428_150821.jsonl', RUN_DIR / 'c5_20260428_150821.judge.jsonl'),
    'C6': (RUN_DIR / 'c6_20260428_164150.jsonl', RUN_DIR / 'c6_20260428_164150.judge.jsonl'),
}

for cfg, (r, j) in PAIRS.items():
    print(f'{cfg}: run={r.name}  judge={j.name}  exists={r.exists() and j.exists()}')

C1: run=c1_20260429_163155.jsonl  judge=c1_20260429_163155.judge.jsonl  exists=True
C2: run=c2_20260429_144635.jsonl  judge=c2_20260429_144635.judge.jsonl  exists=True
C3: run=c3_20260429_081503.jsonl  judge=c3_20260429_081503.judge.jsonl  exists=True
C4: run=c4_20260429_111924.jsonl  judge=c4_20260429_111924.judge.jsonl  exists=True
C5: run=c5_20260428_150821.jsonl  judge=c5_20260428_150821.judge.jsonl  exists=True
C6: run=c6_20260428_164150.jsonl  judge=c6_20260428_164150.judge.jsonl  exists=True


## Step 2 — Preview a sample row 

In [8]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

run_c1 = read_jsonl(PAIRS['C1'][0])
judge_c1 = read_jsonl(PAIRS['C1'][1])

print('--- Run row [0] ---')
print(json.dumps(run_c1[0], ensure_ascii=False, indent=2))
print('\n--- Judge row [0] ---')
print(json.dumps(judge_c1[0], ensure_ascii=False, indent=2))

--- Run row [0] ---
{
  "config": "C1",
  "question_id": "1",
  "question": "Bây giờ thời tiết ở Hà Nội như thế nào?",
  "intent_gold": "current_weather",
  "scope_gold": "city",
  "difficulty": "easy",
  "source": "v1_legacy",
  "expected_tools": "[\"get_current_weather\", \"detect_phenomena\", \"get_clothing_advice\", \"get_rain_timeline\", \"get_hourly_forecast\"]",
  "expected_abstain": "False",
  "expected_clarification": "False",
  "response": "Hiện tại (17:00 Thứ Tư 29/04/2026) ở Hà Nội, thời tiết là **trời mây**, nhiệt độ **nóng 28.1°C**, độ ẩm **50%**, gió **dịu cấp 3 (4.0 m/s), giật 7.2 m/s, hướng Đông Bắc**, mây **rải rác 36%**. Chỉ số UV **thấp - an toàn 0.6**, áp suất **trung bình cao 1011.0 hPa**, tầm nhìn **tốt 10.0 km**.\n\nNếu bạn muốn biết thời tiết trong khung giờ khác như chiều nay, tối nay, hoặc ngày mai, mình có thể tra giúp bạn!",
  "tools_called": [
    "get_current_weather"
  ],
  "tool_outputs": "{\"áp dụng cho\": \"17:00 Thứ Tư 29/04/2026\", \"⚠ snapshot\": t

## Step 3 — Compute metrics per config

In [9]:
def metrics_for(cfg):
    run_path, judge_path = PAIRS[cfg]
    runs = read_jsonl(run_path)
    judges = {r['question_id']: r for r in read_jsonl(judge_path)}

    tool_acc, tool_prec, faith, rel = [], [], [], []
    lat, in_tok, out_tok = [], [], []
    for r in runs:
        intent = r['intent_gold']
        scope = r.get('scope_gold', '')
        tools = r.get('tools_called') or []
        tool_acc.append(1.0 if check_tool_accuracy(intent, tools, scope) else 0.0)
        tool_prec.append(check_tool_precision(intent, tools, scope))
        lat.append(r['total_latency_ms'])
        in_tok.append(r['input_tokens'])
        out_tok.append(r['output_tokens'])
        j = judges.get(r['question_id'])
        if j is not None:
            if j.get('judge_faithfulness_score') is not None:
                faith.append(j['judge_faithfulness_score'])
            if j.get('judge_relevance_score') is not None:
                rel.append(j['judge_relevance_score'])

    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {
        'Tool acc (%)':     round(mean(tool_acc)  * 100, 1),
        'Tool prec (%)':    round(mean(tool_prec) * 100, 1),
        'Faithfulness (%)': round(mean(faith) / 5 * 100, 1),
        'Relevance (%)':    round(mean(rel)   / 5 * 100, 1),
        'Latency (s)':      round(mean(lat) / 1000, 1),
        'Tokens (K)':       round((mean(in_tok) + mean(out_tok)) / 1000, 1),
    }

summary = pd.DataFrame({cfg: metrics_for(cfg) for cfg in PAIRS}).T
summary

,Tool acc (%),Tool prec (%),Faithfulness (%),Relevance (%),Latency (s),Tokens (K)
C1,92.6,90.3,89.5,92.4,17.6,28.9
C2,84.8,83.7,80.7,93.2,19.3,42.6
C3,89.4,86.8,79.2,91.7,18.9,28.5
C4,92.2,89.7,80.1,91.8,18.3,29.3
C5,88.2,81.5,84.7,93.7,11.0,36.6
C6,83.4,79.2,87.2,94.9,11.6,38.8
